In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.colorbar import ColorbarBase
import os

CONTRAST_ORDER = [
    "RBD vs HC",
    "Hyposmia vs HC",
]
CONTRAST_LABELS = {
    "RBD vs HC":      "RBD\nvs HC",
    "Hyposmia vs HC": "Hyposmia\nvs HC",
}

CLINICAL_ORDER = ["moca_z", "updrs3_score_z", "gds_z"]
CLINICAL_LABELS = {"moca_z": "MoCA", "updrs3_score_z": "UPDRS-III", "gds_z": "GDS"}

MAP_ORDER_CSV = [
    "Serotonin | 5HT1a", "Serotonin | 5HT1b", "Serotonin | 5HT2a",
    "Serotonin | 5HT4", "Serotonin | 5HTT",
    "Dopamine | D1", "Dopamine | D23", "Dopamine | DAT", "Dopamine | FDOPA",
    "GABA | GABAa",
    "Glutamate | mGluR5", "Glutamate | NMDA",
    "Noradrenaline/Acetylcholine | NET", "Noradrenaline/Acetylcholine | VAChT",
]
MAP_SHORT = {
    "Serotonin | 5HT1a": "5-HT1a",
    "Serotonin | 5HT1b": "5-HT1b",
    "Serotonin | 5HT2a": "5-HT2a",
    "Serotonin | 5HT4": "5-HT4",
    "Serotonin | 5HTT": "5-HTT",
    "Dopamine | D1": "D1",
    "Dopamine | D23": "D2/3",
    "Dopamine | DAT": "DAT",
    "Dopamine | FDOPA": "FDOPA",
    "GABA | GABAa": "GABA_A",
    "Glutamate | mGluR5": "mGluR5",
    "Glutamate | NMDA": "NMDA",
    "Noradrenaline/Acetylcholine | NET": "NET",
    "Noradrenaline/Acetylcholine | VAChT": "VAChT",
}

SYSTEM_BANDS = [
    (0, 4, "#f0f5fc"),
    (5, 8, "#fdf3ee"),
    (9, 9, "#f5f0fc"),
    (10, 11, "#eefaf2"),
    (12, 13, "#fceef5"),
]
SYSTEM_EDGE_COLORS = {
    "#f0f5fc": "#5b8dd9",
    "#fdf3ee": "#e0834e",
    "#f5f0fc": "#9b72c2",
    "#eefaf2": "#5aab6e",
    "#fceef5": "#d47fa6",
}

VMIN, VMAX = -0.25, 0.25
CMAP = "RdBu_r"

OUT_DIR = "../../results/figures/heatmap"
os.makedirs(OUT_DIR, exist_ok=True)

print("Constants loaded.")

In [ ]:
df_thick = pd.read_csv("../../results/clinical_correlation_thickness_cortical_subgroups.csv")

data_thick = {}
for contrast in CONTRAST_ORDER:
    sub = df_thick[df_thick["contrast"] == contrast]
    r_mat = np.full((len(MAP_ORDER_CSV), len(CLINICAL_ORDER)), np.nan)
    q_mat = np.full((len(MAP_ORDER_CSV), len(CLINICAL_ORDER)), np.nan)
    for mi, m in enumerate(MAP_ORDER_CSV):
        for ci, c in enumerate(CLINICAL_ORDER):
            row = sub[(sub["map"] == m) & (sub["clinical_var"] == c)]
            if len(row) == 1:
                r_mat[mi, ci] = row["spearman_r"].values[0]
                q_mat[mi, ci] = row["spearman_p_fdr"].values[0]
    data_thick[contrast] = {"r": r_mat, "q": q_mat}

print("Thickness data loaded. Shape per contrast:", r_mat.shape)
sig_total = sum((data_thick[c]["q"] < 0.05).sum() for c in CONTRAST_ORDER)
print(f"Total FDR-significant cells (thickness): {sig_total}")

In [ ]:
df_subc = pd.read_csv("../../results/clinical_correlation_volume_subcortical_subgroups.csv")

data_subc = {}
for contrast in CONTRAST_ORDER:
    sub = df_subc[df_subc["contrast"] == contrast]
    r_mat = np.full((len(MAP_ORDER_CSV), len(CLINICAL_ORDER)), np.nan)
    q_mat = np.full((len(MAP_ORDER_CSV), len(CLINICAL_ORDER)), np.nan)
    for mi, m in enumerate(MAP_ORDER_CSV):
        for ci, c in enumerate(CLINICAL_ORDER):
            row = sub[(sub["map"] == m) & (sub["clinical_var"] == c)]
            if len(row) == 1:
                r_mat[mi, ci] = row["spearman_r"].values[0]
                q_mat[mi, ci] = row["spearman_p_fdr"].values[0]
    data_subc[contrast] = {"r": r_mat, "q": q_mat}

print("Subcortical data loaded. Shape per contrast:", r_mat.shape)
sig_total = sum((data_subc[c]["q"] < 0.05).sum() for c in CONTRAST_ORDER)
print(f"Total FDR-significant cells (subcortical): {sig_total}")

In [ ]:
def draw_heatmap_panel(ax, r_mat, q_mat, title=""):
    n_maps, n_clin = r_mat.shape

    for start, end, color in SYSTEM_BANDS:
        ax.axhspan(start - 0.5, end + 0.5, color=color, zorder=0)

    im = ax.imshow(
        r_mat, cmap=CMAP, vmin=VMIN, vmax=VMAX,
        aspect="auto", interpolation="nearest", zorder=1,
    )

    for mi in range(n_maps):
        for ci in range(n_clin):
            q = q_mat[mi, ci]
            if not np.isnan(q) and q < 0.05:
                rect = mpatches.FancyBboxPatch(
                    (ci - 0.5, mi - 0.5), 1, 1,
                    boxstyle="square,pad=0",
                    linewidth=2.2, edgecolor="#222222",
                    facecolor="none", zorder=3,
                )
                ax.add_patch(rect)
                ax.text(
                    ci, mi, "*",
                    ha="center", va="center",
                    fontsize=13, fontweight="bold",
                    color="#111111", zorder=4, clip_on=False,
                )

    for ci in range(n_clin + 1):
        ax.axvline(ci - 0.5, color="white", linewidth=0.8, zorder=2)
    for mi in range(n_maps + 1):
        ax.axhline(mi - 0.5, color="white", linewidth=0.8, zorder=2)

    for start, end, color in SYSTEM_BANDS:
        edge_color = SYSTEM_EDGE_COLORS[color]
        rect = mpatches.FancyBboxPatch(
            (-0.5 - 0.18, start - 0.5), 0.13, end - start + 1,
            boxstyle="square,pad=0",
            linewidth=0, facecolor=edge_color,
            transform=ax.transData, clip_on=False, zorder=5,
        )
        ax.add_patch(rect)

    ax.set_xlim(-0.5, n_clin - 0.5)
    ax.set_ylim(n_maps - 0.5, -0.5)

    ax.set_xticks(range(n_clin))
    ax.set_xticklabels(
        [CLINICAL_LABELS[c] for c in CLINICAL_ORDER],
        fontsize=10, fontweight="bold", rotation=45, ha="right",
    )
    ax.tick_params(axis="x", length=0, pad=4)

    ax.set_yticks(range(n_maps))
    ax.set_yticklabels([MAP_SHORT[m] for m in MAP_ORDER_CSV], fontsize=9)
    ax.tick_params(axis="y", length=0, pad=16)

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)

    return im

In [ ]:
# ── Cortical Thickness Heatmap ─────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 2, figsize=(10, 7),
    gridspec_kw={"wspace": 0.90},
)

for i, contrast in enumerate(CONTRAST_ORDER):
    im = draw_heatmap_panel(
        axes[i],
        data_thick[contrast]["r"],
        data_thick[contrast]["q"],
        title=CONTRAST_LABELS[contrast],
    )
    axes[i].text(-0.05, 1.05, "abcdefghijklmnopqrstuvwxyz"[i],
                 transform=axes[i].transAxes, fontsize=12,
                 fontweight='bold', va='top', clip_on=False)

cbar_ax = fig.add_axes([0.93, 0.18, 0.015, 0.65])
norm = Normalize(vmin=VMIN, vmax=VMAX)
cb = ColorbarBase(cbar_ax, cmap=CMAP, norm=norm, orientation="vertical")
cb.set_label("Spearman r", fontsize=10)
cb.ax.tick_params(labelsize=9)

fig.suptitle(
    "Clinical Correlations — Cortical Thickness Colocalization (Prodromal Subgroups)",
    fontsize=13, fontweight="bold", y=1.01,
)
fig.text(
    0.50, -0.03,
    "* FDR q < 0.05   |   Color = Spearman r (covariate-adjusted)",
    ha="center", fontsize=9, color="#444444",
)

plt.subplots_adjust(left=0.16, right=0.90, top=0.90, bottom=0.15)

out_path = os.path.join(OUT_DIR, "heatmap_clinical_correlation_thickness_subgroups.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out_path}")

In [ ]:
# ── Subcortical Volume Heatmap ─────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 2, figsize=(10, 7),
    gridspec_kw={"wspace": 0.90},
)

for i, contrast in enumerate(CONTRAST_ORDER):
    im = draw_heatmap_panel(
        axes[i],
        data_subc[contrast]["r"],
        data_subc[contrast]["q"],
        title=CONTRAST_LABELS[contrast],
    )
    axes[i].text(-0.05, 1.05, "abcdefghijklmnopqrstuvwxyz"[i],
                 transform=axes[i].transAxes, fontsize=12,
                 fontweight='bold', va='top', clip_on=False)

cbar_ax = fig.add_axes([0.93, 0.18, 0.015, 0.65])
norm = Normalize(vmin=VMIN, vmax=VMAX)
cb = ColorbarBase(cbar_ax, cmap=CMAP, norm=norm, orientation="vertical")
cb.set_label("Spearman r", fontsize=10)
cb.ax.tick_params(labelsize=9)

fig.suptitle(
    "Clinical Correlations — Subcortical Volume Colocalization (Prodromal Subgroups)",
    fontsize=13, fontweight="bold", y=1.01,
)
fig.text(
    0.50, -0.03,
    "* FDR q < 0.05   |   Color = Spearman r (covariate-adjusted)",
    ha="center", fontsize=9, color="#444444",
)

plt.subplots_adjust(left=0.16, right=0.90, top=0.90, bottom=0.15)

out_path = os.path.join(OUT_DIR, "heatmap_clinical_correlation_subcortical_subgroups.png")
fig.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {out_path}")